[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_4_HigherOrder_Operators_AR.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs_ar/Lab_4_HigherOrder_Operators_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏📐 اللاب 4: المعادلات الخطية المتجانسة من الرتب العليا وأصل معادلة لابلاس

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏<i>غرفة التحكم: لماذا تمنع معادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla^2\phi = 0}$</span>‏ وجود أي مصيدة محلية؟</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع في المذكرة:</b> الباب الأول، القسم 1.4 — <b>ترقيم المذكرة ص 34–57</b> <i>(صفحات ملف الـ PDF من 28 إلى 51؛ معادلة لابلاس في ص 39 بالترقيم المكتوب / PDF ص 33)</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; background-color:rgba(128,128,128,0.12); padding:10px 14px; border-radius:6px; overflow-x:auto; margin:10px 0; font-family:monospace; white-space:pre;">‏[اللاب 1–3: الميول، مسارات الانسياب، وهوامش الأمان] ──► 📍 [اللاب 4: معادلة لابلاس والجهد التوافقي] ──► [اللاب 5: المؤثرات القابلة للتحليل ومشكلة الاضمحلال] ──► [اللاب 6: الحل العددي SOR]</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. رمز المذكرة (PDF pp. 28–51)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الهندسي والفيزيائي

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الدالة الفعلية في المشروع (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/</code>‏)

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_x = \frac{\partial}{\partial x}, \quad D_y = \frac{\partial}{\partial y}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏مؤثرات التفاضل الجزئي الجبرية

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تحويل المعادلة التفاضلية إلى كثيرة حدود جبرية

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(D_x^2 + D_y^2)z = 0}$</span>‏ <i>(مثال 5 ص 39 / PDF ص 33)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلة لابلاس (Laplace's Equation):</b> انعدام التقوس الصافي؛ السطح مشدود كغشاء مطاطي

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المعادلة الحاكمة لحقل الجهد الملاحي في <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${m^2 + 1 = 0 \implies m = \pm i}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الجذور المميزة المركبة المترافقة لمعادلة لابلاس

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تثبت أن الحل دالة توافقية ناعمة بلا قمم أو حفر داخلية

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\nabla^2 \phi \approx \frac{\phi_{E}+\phi_{W}+\phi_{S}+\phi_{N} - 4\phi_{C}}{h^2} = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏قيمة أي نقطة تساوي تماماً متوسط جيرانها الأربعة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الدالة <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&#95;avg(phi)</code>‏</a> في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/solvers.py</code>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. الفكرة ببساطة: لماذا نحتاج إلى معادلة من الرتبة الثانية (معادلة لابلاس)؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في اللاب 2 رأينا أن المركبة تتبع منحدر حقل الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi(x, y)}$</span>‏ عبر معادلة لاجرانج (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend</code>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏ولكن <b>كيف نصنع حقل الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi(x,y)}$</span>‏ نفسه</b> بحيث يضمن للمركبة عدم الوقوع في أي "حفرة جهد وهمية" (Local Minimum) قبل وصولها للهدف؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏تخيل غشاءً مطاطياً مشدوداً على إطار:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• إذا ثبتنا ارتفاع الغشاء عند جدران الغرفة (أو العوائق) عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 1}$</span>‏، وثبتنا نقطة الهدف فقط عند القاع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 0}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• فإن الغشاء المطاطي في الفراغ بينهما يحقق معادلة التوازن التام:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)\phi = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• هذه هي <b>معادلة لابلاس (Laplace's Equation)</b>!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• هندسياً، المشتقة الثانية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 \phi}{\partial x^2}}$</span>‏ تقيس التقوس في اتجاه ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏، و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2 \phi}{\partial y^2}}$</span>‏ تقيس التقوس في اتجاه ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• لكي يكون مجموعهما صفراً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla^2 \phi = 0}$</span>‏)، لا يمكن للسطح أن يتقوس للأعلى في الاتجاهين معاً في نفس النقطة! أي <b>يستحيل رياضياً وجود أي قاع محلي (Local Minimum)</b> داخل المجال المفتوح لأن القاع المحلي يتطلب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_{xx} > 0}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_{yy} > 0}$</span>‏ معاً (ومجموع رقمين موجبين لا يمكن أن يساوي صفراً!).

</div>

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab4

%matplotlib inline
print("تم تهيئة بيئة اللاب 4 بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. التغطية الكاملة للقسم 1.4 في المذكرة: المعادلات المتجانسة وجميع حالات التكامل الخاص الـ 6 (PDF ص 28–51)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نسمي المعادلة التفاضلية الجزئية الخطية <b>متجانسة في الرتبة (Homogeneous of order ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${n}$</span>‏)</b> إذا كانت جميع المشتقات فيها من نفس الرتبة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${n}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(a_0 D_x^n + a_1 D_x^{n-1} D_y + \dots + a_n D_y^n) z = f(x, y) \quad \Longleftrightarrow \quad F(D_x, D_y) z = f(x, y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏يتكون الحل العام من جزأين: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = z_1\text{ (الحل المكمل C.F.)} + z_2\text{ (التكامل الخاص P.I.)}}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏أولاً: إيجاد الحل المكمل (Complementary Function ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_1}$</span>‏ — ص 35–39 / PDF ص 29–33)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نفرض حلاً من الشكل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = \phi(y + mx)}$</span>‏، فتكون ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x z = m\phi'}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y z = \phi'}$</span>‏. بالتعويض في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y)z = 0}$</span>‏ نحصل على <b>المعادلة المساعدة (Auxiliary Equation)</b> بوضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x = m}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y = 1}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$F(m, 1) = a_0 m^n + a_1 m^{n-1} + \dots + a_n = 0$$

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏حالة جذور المعادلة المساعدة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F(m, 1) = 0}$</span>‏

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏شكل الحل المكمل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_1}$</span>‏ في المذكرة

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 1: جذور حقيقية ومختلفة</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${m_1 \neq m_2}$</span>‏ <i>(ص 36 / PDF ص 30)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_1 = \phi_1(y + m_1 x) + \phi_2(y + m_2 x)}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 2: جذور مكررة (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${k}$</span>‏ مرات)</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${m_1 = \dots = m_k = m}$</span>‏ <i>(ص 37 / PDF ص 31)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_1 = \phi_1(y + mx) + x\,\phi_2(y + mx) + \dots + x^{k-1}\phi_k(y + mx)}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 3: جذور مركبة</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${m = \alpha \pm i\beta}$</span>‏ <i>(ص 39 / PDF ص 33)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_1 = \phi_1(y + (\alpha + i\beta)x) + \phi_2(y + (\alpha - i\beta)x)}$</span>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏📖 المثال المحوري في المذكرة: معادلة لابلاس (مثال 5 ص 39 / PDF ص 33)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> حل معادلة لابلاس ثنائية الأبعاد:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{\partial^2 z}{\partial x^2} + \frac{\partial^2 z}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>المعادلة المساعدة:</b> نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x = m}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y = 1}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$m^2 + 1 = 0 \implies m^2 = -1 \implies m = \pm i$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>الحل العام التحليلي:</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = \phi_1(y + ix) + \phi_2(y - ix)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_1, \phi_2}$</span>‏ دالتان اختياريتان. (وإذا اخترنا مثلاً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_1(w) = \frac{1}{2i}\sin(\pi i w)}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_2(w) = -\frac{1}{2i}\sin(\pi i w)}$</span>‏، نحصل على الدالة التوافقية الحقيقية الشهيرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(\pi x)\sinh(\pi y)}$</span>‏!).

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏ثانياً: إيجاد التكامل الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2 = \frac{1}{F(D_x, D_y)} f(x, y)}$</span>‏ — الطريقة العامة والحالات الست المختصرة (PDF ص 34–51)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما يكون الطرف الأيمن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) \neq 0}$</span>‏، تشرح المذكرة <b>طريقة عامة (General Method)</b> بالإضافة إلى <b>6 طرق مختصرة (Short Methods)</b> بحسب شكل الدالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y)}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🧭 جدول القرار السريع لحالات التكامل الخاص الست (PDF ص 34–51)

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏الحالة في المذكرة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏شكل الطرف الأيمن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(x, y)}$</span>‏

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏القاعدة السريعة لحساب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_2 = \frac{1}{F(D_x, D_y)}f(x,y)}$</span>‏

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏ماذا لو صفر المقام (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F = 0}$</span>‏)؟

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الطريقة العامة</b> <i>(ص 40 / PDF ص 34)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏أي دالة عامة (مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\sec^2(x+y)}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏حلل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F = \prod(D_x - m_i D_y)}$</span>‏ وحل كل عامل بمعادلة لاجرانج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{dx}{1} = \frac{dy}{-m} = \frac{du}{f}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تعمل دائماً بدون استثناء!

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 1</b> <i>(ص 44 / PDF ص 38)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${e^{ax + by}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏استبدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_x \to a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_y \to b}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_2 = \frac{1}{F(a,b)}e^{ax+by}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F = (D_x - \frac{a}{b}D_y)^r G}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_2 = \frac{1}{G(a,b)}\frac{x^r}{r!}e^{ax+by}}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 2</b> <i>(ص 47 / PDF ص 41)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\sin(ax+by)}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\cos(ax+by)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏استبدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_x^2 \to -a^2,\; D_x D_y \to -ab,\; D_y^2 \to -b^2}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏حوّل لأسية بصيغة أويلر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{e^{i\theta} \pm e^{-i\theta}}{2}}$</span>‏ وطبق رنين الحالة 1

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 3</b> <i>(ص 51 / PDF ص 45)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏كثيرة حدود ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x^a y^b}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏اسحب أصغر قوة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_x}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_y}$</span>‏) عامل مشترك وفك ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{1}{1-\theta} = 1 + \theta + \theta^2 + \dots}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏التكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{1}{D_x}}$</span>‏ يعني التكامل بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 4</b> <i>(ص 53 / PDF ص 47)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${e^{ax+by} V(x,y)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الإزاحة الأسية:</b> أخرج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${e^{ax+by}}$</span>‏ خارج المؤثر واستبدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${D_x \to D_x+a,\; D_y \to D_y+b}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏طبّق الحالة 1 أو 2 أو 3 على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${V(x,y)}$</span>‏ بعد الإزاحة

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 5 و 6</b> <i>(ص 56 / PDF ص 50)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏أي دالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${g(ax+by)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏إذا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F(a,b)\neq 0}$</span>‏: كامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${g(u)}$</span>‏ عدد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${n}$</span>‏ مرات: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{1}{F(a,b)}\int\dots\int g(u)du^n}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحالة 6:</b> لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(bD_x - aD_y)^n z = g(ax+by)}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z_2 = \frac{x^n}{n!\,b^n}g(ax+by)}$</span>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 شرح الطريقة العامة (General Method via Lagrange — ص 40–44 / PDF ص 34–38)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>متى نستخدمها؟</b> عندما لا تكون ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y)}$</span>‏ من الدوال القياسية البسيطة (أو عندما نريد الاشتقاق من المبادئ الأولى).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الفكرة:</b> نحلل المؤثر إلى عوامل خطية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y) = (D_x - m_1 D_y)(D_x - m_2 D_y)}$</span>‏، ونفرض ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u_1 = (D_x - m_1 D_y)z}$</span>‏، فنحصل على معادلة من الرتبة الأولى:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(D_x - m_2 D_y)u_1 = f(x, y) \iff \frac{\partial u_1}{\partial x} - m_2 \frac{\partial u_1}{\partial y} = f(x, y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نحلها بمعادلات لاجرانج المساعدة من <b>اللاب 2</b>: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{1} = \frac{dy}{-m_2} = \frac{du_1}{f(x,y)}}$</span>‏، ثم نكرر نفس الخطوة لإيجاد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>📖 مثال 1 في المذكرة (ص 41–43 / PDF ص 35–37):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y^2)z = \sec^2(x + y)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نحلل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x + D_y)(D_x - D_y)z = \sec^2(x + y)}$</span>‏. وبحل العاملين بالتتابع عبر لاجرانج نحصل على التكامل الخاص:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{x}{2}\tan(x + y)}$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الحالة 1: الدالة الأسية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) = e^{ax + by}}$</span>‏ وحالة الرنين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) = 0}$</span>‏ (ص 44–47 / PDF ص 38–41)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>أ) إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) \neq 0}$</span>‏:</b> نعوض مباشرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x \to a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y \to b}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{1}{F(a, b)}e^{ax + by}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>مثال 2 (ص 45 / PDF ص 39):</b> في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_x D_y - 6D_y^2)z = e^{2x - 3y}}$</span>‏، لدينا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 2, b = -3}$</span>‏، فيكون ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(2, -3) = 4 - (2)(-3) - 6(-3)^2 = 4 + 6 - 54 = -44 \neq 0 \implies \boxed{z_2 = -\frac{1}{44}e^{2x - 3y}}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>ب) ماذا لو كان المقام يساوي صفراً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) = 0}$</span>‏ (حالة الرنين / الفشل)؟ (ص 45–47 / PDF ص 39–41)</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) = 0}$</span>‏، فهذا يعني أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\left(D_x - \frac{a}{b}D_y\right)}$</span>‏ عامل من عوامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y)}$</span>‏ مكرر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${r}$</span>‏ من المرات:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$F(D_x, D_y) = \left(D_x - \frac{a}{b}D_y\right)^r G(D_x, D_y), \qquad G(a, b) \neq 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نعوض بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ في الجزء غير الصفري ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${G(a, b)}$</span>‏ فقط، أما العامل الصفري المكرر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${r}$</span>‏ مرات فيتحول إلى <b>ضرب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{x^r}{r!}}$</span>‏</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{1}{G(a, b)} \frac{x^r}{r!} e^{ax + by}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>📖 مثال 3 في المذكرة (ص 46 / PDF ص 40):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_x D_y - 6D_y^2)z = e^{3x + y}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>1.</b> هنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 3, b = 1}$</span>‏. لو عوضنا مباشرة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(3, 1) = 9 - 3 - 6 = 0}$</span>‏! (حالة رنين لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{3x+y} = e^{3(x + y/3)}}$</span>‏ هي أصلًا ضمن الحل المكمل!).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>2.</b> نحلل المؤثر: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 - D_x D_y - 6D_y^2 = (D_x - 3D_y)^1 (D_x + 2D_y)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>3.</b> إذن العامل الصفري هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - 3D_y)}$</span>‏ بتكرار ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${r = 1}$</span>‏، والجزء غير الصفري هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${G(D_x, D_y) = D_x + 2D_y}$</span>‏ حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${G(3, 1) = 3 + 2(1) = 5}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>4.</b> نطبق القانون مباشرة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{z_2 = \frac{1}{5}\frac{x^1}{1!}e^{3x + y} = \frac{x}{5}e^{3x + y}}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>تنبيه خطأ مطبعي في المذكرة (ص 49، مثال 5 / PDF ص 43):</b><br>
‏في السطر الأخير من صفحة 49 بالترقيم المكتوب (PDF ص 43) عند حساب التكامل الخاص للدالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{2x+3y}}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a=2, b=3}$</span>‏) للمؤثر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F = D_x^2 - 3D_x D_y + 2D_y^2}$</span>‏، كتبت المذكرة بالخطأ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(1,1) = 4 - 18 + 18 = 4}$</span>‏. الأرقام المحسوبة (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${2^2 - 3(2)(3) + 2(3)^2 = 4 - 18 + 18 = 4}$</span>‏) صحيحة تماماً لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a,b)=(2,3)}$</span>‏، لكن الرمز المكتوب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(1,1)}$</span>‏ هو خطأ مطبعي والصواب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{F(2, 3) = 4}}$</span>‏!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الحالة 2: دوال الجيب وجيب التمام ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(ax+by)}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos(ax+by)}$</span>‏ (ص 47–51 / PDF ص 41–45)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>أ) في المؤثرات من الرتبة الثانية (بشرط عدم انعدام المقام):</b> نستبدل كل:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{D_x^2 \to -a^2, \qquad D_x D_y \to -ab, \qquad D_y^2 \to -b^2}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(تنبيه الإشارات: السالب خارج التربيع دائماً! فإذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a=2, b=-3}$</span>‏ فإن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 \to -4}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2 \to -9}$</span>‏ بينما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x D_y \to -(2)(-3) = +6}$</span>‏).</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>ب) ماذا لو صفر المقام في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos}$</span>‏؟ (مثال 6 ص 50–51 / PDF ص 44–45):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 + D_y^2)z = \cos(mx + my)}$</span>‏، لدينا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = m, b = m}$</span>‏، فالتعويض يعطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-m^2 + (-m^2)}$</span>‏ أو في حالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y^2)z = \cos(mx+my)}$</span>‏ يعطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-m^2 - (-m^2) = 0}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما ينعدم المقام (أو في المؤثرات من رتب فردية)، تضع المذكرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos\theta = \frac{e^{i\theta} + e^{-i\theta}}{2}}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin\theta = \frac{e^{i\theta} - e^{-i\theta}}{2i}}$</span>‏) وتطبق قاعدة الرنين من <b>الحالة 1</b>!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الحالة 3: كثيرات الحدود ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) = x^a y^b}$</span>‏ بطريقة المفكوك (ص 51–53 / PDF ص 45–47)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الفكرة:</b> نسحب العامل ذا أصغر أس لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y}$</span>‏) من المقام ليصبح المقام على الصورة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^k [1 - \theta(D_x, D_y)]}$</span>‏، ثم نرفعه للبسط بمتسلسلة ذات الحدين (المتسلسلة الهندسية):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{\frac{1}{1 - \theta} = 1 + \theta + \theta^2 + \theta^3 + \dots, \qquad \frac{1}{1 + \theta} = 1 - \theta + \theta^2 - \theta^3 + \dots}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لماذا تتوقف المتسلسلة ولا تستمر إلى ما لا نهاية؟ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x^a y^b}$</span>‏ كثيرة حدود، فمشتقاتها العليا تصبح <b>صفراً</b> بعد عدد محدود من الخطوات! وتذكر أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{D_x}}$</span>‏ معناه <b>التكامل بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏</b> (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\int (\cdots)\,dx}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>📖 مثال 7 في المذكرة (ص 52 / PDF ص 46):</b> أوجد التكامل الخاص لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - 2D_x D_y)z = x^3 y}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>1.</b> نسحب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2}$</span>‏ عاملاً مشتركاً من المقام:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{D_x^2\left(1 - \frac{2D_y}{D_x}\right)}(x^3 y) = \frac{1}{D_x^2}\left(1 - \frac{2D_y}{D_x}\right)^{-1}(x^3 y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>2.</b> نفك القوس بمتسلسلة هندسية (نحتاج فقط حتى المشتقة الأولى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y}$</span>‏ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2(x^3 y) = 0}$</span>‏!):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{D_x^2}\left(1 + \frac{2D_y}{D_x} + \dots\right)(x^3 y) = \frac{1}{D_x^2}\left[x^3 y + \frac{2}{D_x}\underbrace{D_y(x^3 y)}_{x^3}\right]$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>3.</b> نكامل بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{D_x}(x^3) = \frac{x^4}{4}}$</span>‏، ثم نكامل مرتين لوجود ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{D_x^2}}$</span>‏ خارج القوس):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{D_x^2}\left(x^3 y + \frac{x^4}{2}\right) = \frac{1}{D_x}\left(\frac{x^4 y}{4} + \frac{x^5}{10}\right) = \boxed{\frac{x^5 y}{20} + \frac{x^6}{60}}$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الحالة 4: حاصل ضرب دالة أسية في أي دالة أخرى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) = e^{ax + by} V(x, y)}$</span>‏ (ص 53–55 / PDF ص 47–50)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>نظرية الإزاحة الأسية (Exponential Shift Rule):</b> يمكنك إخراج الدالة الأسية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{ax + by}}$</span>‏ إلى يسار المؤثر الكسري فوراً بشرط <b>إزاحة كل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x \to D_x + a}$</span>‏ وكل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y \to D_y + b}$</span>‏</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{\frac{1}{F(D_x, D_y)}\left[e^{ax + by} V(x, y)\right] = e^{ax + by} \frac{1}{F(D_x + a, D_y + b)} V(x, y)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وبعد إخراج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{ax+by}}$</span>‏، تصبح المسألة مجرد تأثير على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${V(x,y)}$</span>‏ فنحلها بالحالة 1 أو 2 أو 3!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>📖 مثال 10 في المذكرة (ص 54 / PDF ص 48):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y)^2 z = x e^{x + y}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>1.</b> نخرج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{x + y}}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 1, b = 1}$</span>‏) ونزيح ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x \to D_x + 1}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y \to D_y + 1}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(D_x + 1) - (D_y + 1) = D_x - D_y \implies z_2 = e^{x+y}\frac{1}{(D_x - D_y)^2}(x)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>2.</b> الآن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${V(x,y) = x^1 y^0}$</span>‏ لا يعتمد على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ إطلاقاً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y(x) = 0}$</span>‏)، فيختزل المؤثر إلى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{D_x^2}(x)}$</span>‏ (تكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ مرتين بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = e^{x + y}\frac{1}{D_x^2}(x) = \frac{x^3}{6}e^{x + y}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(وكذلك في <b>مثال 11 و 12 ص 54–55 / PDF ص 48–50</b> تطبق المذكرة الإزاحة الأسية مع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(2x+3y)}$</span>‏ ومع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x^2 y}$</span>‏ بنفس الخطوات تماماً!).</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الحالتان 5 و 6: عندما يكون الطرف الأيمن دالة عامة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) = g(ax + by)}$</span>‏ (ص 56–57 / PDF ص 50–51)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الحالة 5 (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) \neq 0}$</span>‏ — ص 56 / PDF ص 50):</b> نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u = ax + by}$</span>‏ ونكامل الدالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${g(u)}$</span>‏ عدد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${n}$</span>‏ من المرات (حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${n}$</span>‏ رتبة المؤثر):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{1}{F(a, b)} \underbrace{\int \dots \int}_{n\text{ مرات}} g(u)\,du^n}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الحالة 6 (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) = 0}$</span>‏ حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y) = (b D_x - a D_y)^n}$</span>‏ — ص 56–57 / PDF ص 50–51):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا كان المؤثر من الصورة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(b D_x - a D_y)^n z = g(ax + by)}$</span>‏، فإن التعويض بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ يعطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(ba - ab)^n = 0}$</span>‏! في هذه الحالة لا نكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${g}$</span>‏ أصلاً، بل نضرب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{x^n}{n!\,b^n}}$</span>‏ مباشرة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{x^n}{n!\,b^n}\,g(ax + by)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>📖 مثال 14 و 15 في المذكرة (ص 57 / PDF ص 51):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ في <b>مثال 14:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2D_x - 3D_y)^2 z = \tan(3x + 2y)}$</span>‏، هنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 3, b = 2, n = 2}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b D_x - a D_y = 2D_x - 3D_y \implies F(3, 2) = 0}$</span>‏. بتطبيق قانون الحالة 6 فوراً:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{x^2}{2!\cdot 2^2}\tan(3x + 2y) = \frac{x^2}{8}\tan(3x + 2y)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ في <b>مثال 15:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - 2D_y)^3 z = (2x + y)^2}$</span>‏، هنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 2, b = 1, n = 3 \implies \boxed{z_2 = \frac{x^3}{3!\cdot 1^3}(2x + y)^2 = \frac{x^3}{6}(2x + y)^2}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>تنبيه خطأ مطبعي في المذكرة (ص 57، مثال 14 / PDF ص 51):</b><br>
‏في حل المثال 14 في المذكرة، كُتب رمز المؤثر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_x) = (2D_x - 3D_y)^2}$</span>‏ بحرف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏ مرتين بين القوسين، وهو خطأ مطبعي بسيط والصواب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{F(D_x, D_y)}}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. 📝 تحدي الورقة والقلم 1 (جذور معادلة لابلاس — مثال 5 ص 39 / PDF ص 33)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> أوجد الجذور المميزة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${m_1, m_2}$</span>‏ للمعادلة المساعدة لمعادلة لابلاس:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(D_x^2 + D_y^2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في بايثون، تُكتب الوحدة التخيلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${i = \sqrt{-1}}$</span>‏ على الصورة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">1j</code>‏، وتُكتب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-i}$</span>‏ على الصورة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">-1j</code>‏.

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏ضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x = m}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y = 1}$</span>‏ فتحصل على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${m^2 + 1 = 0 \implies m = \pm i}$</span>‏. في بايثون اكتب ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">1j</code>‏ و ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">-1j</code>‏.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: جذور معادلة لابلاس
### START CODE HERE ### (≈ 2 lines of code)
# أدخل الجذور التخيلية لمعادلة لابلاس في بايثون:
m1 = None
m2 = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_1(m1, m2)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 📝 تحدي الورقة والقلم 2 (التكامل الخاص P.I. — مثال 4 ص 48 / PDF ص 42)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة (من المذكرة ص 48 / PDF ص 42):</b> أوجد التكامل الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2}$</span>‏ للمعادلة التفاضلية الجزئية غير المتجانسة في الطرف الأيمن:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(D_x^2 - D_x D_y - 6 D_y^2)z = \sin(2x - 3y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>خطوات الحل على ورقتك:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> حدد معاملات الزاوية في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(ax + by) = \sin(2x - 3y)}$</span>‏: لدينا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 2}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = -3}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> عوض في المؤثر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{1}{D_x^2 - D_x D_y - 6 D_y^2}}$</span>‏ باستخدام قاعدة الحالة الثانية:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 \to -a^2}$</span>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x D_y \to -ab}$</span>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2 \to -b^2}$</span>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> احسب قيمة المقام العددي الناتج، ثم اكتب صيغة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2}$</span>‏.

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1 (حساب البدائل الثلاثة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بما أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 2}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = -3}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 \to -(2)^2 = -4}$</span>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x D_y \to -(2)(-3) = +6}$</span>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2 \to -(-3)^2 = -9}$</span>‏

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2 (التعويض في المقام)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عوض بالأرقام في المقام ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2) - (D_x D_y) - 6(D_y^2)}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(-4) - (6) - 6(-9) = -10 + 54 = 44$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذن التكامل الخاص هو مقلوب هذا العدد مضروباً في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(2x - 3y)}$</span>‏.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: التكامل الخاص P.I.
### START CODE HERE ### (≈ 1 line of code)
# اكتب صيغة التكامل الخاص بدلالة x و y:
student_pi = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_2(student_pi)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 💻 التطبيق البرمجي ببايثون: حلال الجذور المميزة للمؤثرات التفاضلية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لحل أي معادلة متجانسة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a_0 D_x^n + a_1 D_x^{n-1}D_y + \dots + a_n D_y^n)z = 0}$</span>‏ برمجياً، نمرر معاملات المعادلة المساعدة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">[a&#95;0, a&#95;1, ..., a&#95;n]</code>‏ إلى دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">np.roots(coeffs)</code>‏ لإيجاد الجذور ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${m_1, \dots, m_n}$</span>‏ وتصنيف نوع المعادلة.

</div>

</div>


In [ ]:
# دالة تقييم برمجية: solve_operator_roots
def solve_operator_roots(coeffs: list[float]) -> list[complex]:
    # تحسب جذور المعادلة المساعدة F(m, 1) = 0 لقائمة المعاملات coeffs
    ### START CODE HERE ### (≈ 1 line of code)
    # استخدم np.roots(coeffs) لحساب الجذور:
    roots = None
    ### END CODE HERE ###
    return list(roots)

# تشغيل الاختبار الآلي:
test_lab4.test_characteristic_roots_solver(solve_operator_roots)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 🎨 الجسر إلى اللاب 6: كيف تتحول معادلة لابلاس التحليلية إلى دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&#95;avg(phi)</code>‏ في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏قد تتساءل: <b>إذا كان لدينا الحل التحليلي العام لمعادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = \phi_1(y + ix) + \phi_2(y - ix)}$</span>‏، فلماذا نحتاج إلى حلال عددي في اللاب 6؟</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• على مربع فارغ منتظم، يمكننا كتابة حل تحليلي مغلق مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi(x, y) = \frac{\sin(\pi x)\sinh(\pi y)}{\sinh(\pi)}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• لكن على خريطة المريخ، توجد <b>مئات الصخور والحفر بأشكال عشوائية غير منتظمة</b>! يستحيل إيجاد صيغة تحليلية واحدة تحقق شروط الحدود حول 10,000 عقدة صخرية عشوائية.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• الحل العبقري: عندما نقرب المشتقات الثانية على شبكة بكسل متساوية المسافات (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\Delta x = \Delta y = h}$</span>‏):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} \approx \frac{\phi_{i,j+1} + \phi_{i,j-1} + \phi_{i+1,j} + \phi_{i-1,j} - 4\phi_{i,j}}{h^2} = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏فإن معادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 + D_y^2)\phi = 0}$</span>‏ تعني ببساطة أن <b>قيمة كل نقطة داخلية تساوي متوسط جيرانها الأربعة (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&#95;avg(phi)</code>‏)</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i,j} = \frac{1}{4}\left(\phi_{\text{North}} + \phi_{\text{South}} + \phi_{\text{East}} + \phi_{\text{West}}\right)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لترى السطح التحليلي التوافقي وتتحقق بنفسك أن دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&#95;avg</code>‏ الفعلية من ملف <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/solvers.py</code>‏</a> تطابق السطح التحليلي!

</div>

</div>


In [ ]:
from rover_path_finder.solvers import _avg

x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)

# نختبر خاصية المتوسط الحسابي للجيران الأربعة على النقاط الداخلية
neighbor_diff = np.abs(_avg(Z)[1:-1, 1:-1] - Z[1:-1, 1:-1])

fig = plt.figure(figsize=(13, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
ax1.set_title(r"1. Harmonic Surface $\nabla^2\phi = 0$ (Zero Local Minima)", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel(r"$\phi(x, y)$")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
im = ax2.imshow(neighbor_diff, origin='lower', extent=[x[1], x[-2], y[1], y[-2]], cmap='magma')
plt.colorbar(im, ax=ax2, label=r"$|\mathrm{\_avg}(\phi) - \phi|$")
ax2.set_title("2. Bridge to solvers.py: 4-Neighbor Average Residual", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")

plt.tight_layout()
plt.show()

print(f"✅ أقصى فرق بين السطح التحليلي ومتوسط الجيران الأربعة _avg(Z): {neighbor_diff.max():.6f}")
print("📌 ملخص اللاب 4:")
print("  1) معادلة لابلاس (D_x^2 + D_y^2)z = 0 لها جذور مميزة مركبة m = ±i وحلولها دوال توافقية ملساء.")
print("  2) الدوال التوافقية تحقق خاصية المتوسط الحسابي، لذا يستحيل أن تحتوي على قاع محلي يحبس المركبة.")
